In [77]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import ExtraTreesClassifier

In [78]:
train = pd.read_csv('data/cleaned/1year/1year_train.csv')
validation = pd.read_csv('data/cleaned/1year/1year_validation.csv')
test = pd.read_csv('data/cleaned/1year/1year_test.csv')

X_train = train.drop(columns='bankrupt')
y_train = train['bankrupt']
X_validation = validation.drop(columns='bankrupt')
y_validation = validation['bankrupt']
X_test = validation.drop(columns='bankrupt')
y_test = validation['bankrupt']

print('Training rows:', len(train))
print('Validation rows:', len(validation))
print('Test rows:', len(test))
print('Bankrupt companies in validation:', int(y_validation.sum()))
print('Bankrupt companies in test:', int(y_test.sum()))

Training rows: 4861
Validation rows: 1042
Test rows: 1042
Bankrupt companies in validation: 41
Bankrupt companies in test: 41


In [79]:
def scores(predictions, probabilities):
    return {
        'Accuracy': accuracy_score(y_validation, predictions),
        'Precision': precision_score(y_validation, predictions, zero_division=0),
        'Recall': recall_score(y_validation, predictions, zero_division=0),
        'F1': f1_score(y_validation, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(y_validation, probabilities),
        'Average precision': average_precision_score(y_validation, probabilities)
    }

Logistic Regression

In [80]:
logistic = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42)
)
param_grid = {
    'logisticregression__max_iter': [100,1000,2500,5000],
    'logisticregression__C' : np.logspace(-4,4,20),
}

log_grid_search = GridSearchCV(logistic, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
log_grid_search.fit(X_train, y_train)

print("Best Parameters:", log_grid_search.best_params_)
print("Best Estimator:", log_grid_search.best_estimator_)

Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best Parameters: {'logisticregression__C': np.float64(0.0001), 'logisticregression__max_iter': 100}
Best Estimator: Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(C=np.float64(0.0001), random_state=42))])


In [81]:
logReg_updated = log_grid_search.best_estimator_
logReg_updated.fit(X_train, y_train)
lgReg_updated_predictions = logReg_updated.predict(X_validation)

lgReg_results = pd.DataFrame({
    'Logistic regression': scores(lgReg_updated_predictions, logReg_updated.predict_proba(X_validation)[:, 1]),
}).T
lgReg_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision
Logistic regression,0.961,0.0,0.0,0.0,0.583,0.084


Class-Weighted Trees

In [82]:
weightedTrees = DecisionTreeClassifier(class_weight='balanced',random_state=42)


param_grid = {
    'splitter': ["best", "random"],
    'max_features': ["sqrt", "log2", None],
    'max_depth': [3,5,8,11,14,17,20],
}

weight_grid_search = GridSearchCV(weightedTrees, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
weight_grid_search.fit(X_train, y_train)

print("Best Parameters:", weight_grid_search.best_params_)
print("Best Estimator:", weight_grid_search.best_estimator_)

Fitting 5 folds for each of 42 candidates, totalling 210 fits
Best Parameters: {'max_depth': 20, 'max_features': None, 'splitter': 'best'}
Best Estimator: DecisionTreeClassifier(class_weight='balanced', max_depth=20, random_state=42)


In [83]:
wTrees_updated = weight_grid_search.best_estimator_
wTrees_updated.fit(X_train, y_train)
wTrees_updated_predictions = wTrees_updated.predict(X_validation)

weightTrees_results = pd.DataFrame({
    'Weighted tree': scores(wTrees_updated_predictions, wTrees_updated.predict_proba(X_validation)[:, 1]),
}).T
weightTrees_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision
Weighted tree,0.957,0.447,0.415,0.43,0.698,0.226


Random forest

In [84]:
randomForest = RandomForestClassifier(random_state=42)


param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ["sqrt", "log2", None],
    'max_depth': [3,5,8,11,14,17,20],
}

rf_grid_search = GridSearchCV(randomForest, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
rf_grid_search.fit(X_train, y_train)

print("Best Parameters:", rf_grid_search.best_params_)
print("Best Estimator:", rf_grid_search.best_estimator_)

Fitting 5 folds for each of 105 candidates, totalling 525 fits
Best Parameters: {'max_depth': 17, 'max_features': None, 'n_estimators': 100}
Best Estimator: RandomForestClassifier(max_depth=17, max_features=None, random_state=42)


In [85]:
randomForest_updated = rf_grid_search.best_estimator_
randomForest_updated.fit(X_train, y_train)
rF_updated_predictions = randomForest_updated.predict(X_validation)

rfu_results = pd.DataFrame({
    'Random forest': scores(rF_updated_predictions, randomForest_updated.predict_proba(X_validation)[:, 1]),
}).T
rfu_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision
Random forest,0.974,0.938,0.366,0.526,0.886,0.591


Extra trees

In [86]:
extra_trees = ExtraTreesClassifier(random_state=42)


param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ["sqrt", "log2", None],
    'max_depth': [3,5,8,11,14,17,20],
}

exTrees_grid_search = GridSearchCV(extra_trees, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
exTrees_grid_search.fit(X_train, y_train)

print("Best Parameters:", exTrees_grid_search.best_params_)
print("Best Estimator:", exTrees_grid_search.best_estimator_)

Fitting 5 folds for each of 105 candidates, totalling 525 fits
Best Parameters: {'max_depth': 17, 'max_features': 'sqrt', 'n_estimators': 100}
Best Estimator: ExtraTreesClassifier(max_depth=17, random_state=42)


In [87]:
exTrees_updated = exTrees_grid_search.best_estimator_
exTrees_updated.fit(X_train, y_train)
extraTrees_updated_predictions = exTrees_updated.predict(X_validation)

exTrU_results = pd.DataFrame({
    'Extra trees': scores(extraTrees_updated_predictions, exTrees_updated.predict_proba(X_validation)[:, 1]),
}).T
exTrU_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision
Extra trees,0.962,1.0,0.024,0.048,0.762,0.164


Final Results

In [88]:
final_results = pd.concat([
    lgReg_results,
    weightTrees_results,
    rfu_results,
    exTrU_results
])

final_results['False positives'] = [
    confusion_matrix(y_validation, lgReg_updated_predictions, labels=[0, 1])[0, 1],
    confusion_matrix(y_validation, wTrees_updated_predictions, labels=[0, 1])[0, 1],
    confusion_matrix(y_validation, rF_updated_predictions, labels=[0, 1])[0, 1],
    confusion_matrix(y_validation, extraTrees_updated_predictions, labels=[0, 1])[0, 1],
]

final_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,Average precision,False positives
Logistic regression,0.961,0.000,0.000,0.000,0.583,0.084,0
Weighted tree,0.957,0.447,0.415,0.430,0.698,0.226,21
Random forest,0.974,0.938,0.366,0.526,0.886,0.591,1
Extra trees,0.962,1.000,0.024,0.048,0.762,0.164,0
